# 🚀 Hướng Dẫn Chạy Huấn Luyện YOLO11n vs YOLO11n+CBAM Trên Google Colab
### Dự án: KLCN-2026 | Phát hiện khuyết tật MVTec AD 15 Categories (20 Epochs)

> **Lưu ý quan trọng:**
> 1. Hãy chọn Runtime: **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).
> 2. Toàn bộ trọng số checkpoint của **mỗi epoch (`save_period=1`)** và biểu đồ đánh giá sẽ được tự động sao lưu trực tiếp vào **Google Drive** của bạn.

## Bước 1: Kết nối Google Drive cá nhân
*Giúp lưu trữ vĩnh viễn checkpoints (`epoch1.pt`, `epoch2.pt`,..., `best.pt`) không bị mất khi Colab ngắt kết nối.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ trên Google Drive cá nhân
!mkdir -p /content/drive/MyDrive/KLCN-2026/runs/baseline
!mkdir -p /content/drive/MyDrive/KLCN-2026/runs/cbam
!mkdir -p /content/drive/MyDrive/KLCN-2026/evaluation_results

## Bước 2: Clone Source Code từ GitHub & Cài đặt Thư viện

In [ ]:
# Clone project từ repo chính thức
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git
%cd yolo11n-cbam-mvtec-defect-detection

# Cài đặt các gói phụ thuộc
!pip install -q ultralytics albumentations tabulate

## Bước 3: Giải nén Dataset vào Colab
*Tải file `mvtec_augmented.zip` lên Google Drive (ví dụ tại `MyDrive/KLCN-2026/mvtec_augmented.zip`) và giải nén siêu tốc vào ổ SSD Colab:*

In [ ]:
!mkdir -p data/processed/split_70_15_15_augmented
!unzip -q /content/drive/MyDrive/KLCN-2026/mvtec_augmented.zip -d data/processed/split_70_15_15_augmented/
!ls -la data/processed/split_70_15_15_augmented/

## Bước 4A: Huấn Luyện MÔ HÌNH BASELINE (YOLO11n Gốc)
*Dành cho thành viên được phân công chạy nhánh Baseline (20 epochs)*

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --model configs/models/yolo11n_baseline.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project experiments/baseline \
    --name exp0_baseline_testing \
    --drive_backup /content/drive/MyDrive/KLCN-2026/runs/baseline/exp0_baseline_testing

## Bước 4B: Huấn Luyện MÔ HÌNH CHÍNH (YOLO11n + CBAM@Backbone)
*Dành cho thành viên được phân công chạy nhánh CBAM (20 epochs)*

In [ ]:
!python src/training/train_cbam.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project experiments/cbam \
    --name exp1_cbam_testing \
    --drive_backup /content/drive/MyDrive/KLCN-2026/runs/cbam/exp1_cbam_testing

## Bước 5: Đánh Giá So Sánh & Xuất Biểu Đồ Tự Động
*Chạy sau khi đã có kết quả của cả 2 mô hình (hoặc trỏ tới thư mục Drive đã lưu):*

In [ ]:
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLCN-2026/runs/baseline/exp0_baseline_testing \
    --cbam_dir /content/drive/MyDrive/KLCN-2026/runs/cbam/exp1_cbam_testing \
    --output_dir /content/drive/MyDrive/KLCN-2026/evaluation_results